In [3]:
# ==========================================
# 1. SYSTEM SETUP: INSTALL ERLANG
# ==========================================
# Update the package list
!apt-get update

# Install the base Erlang packages
!apt-get install -y erlang-base erlang-crypto

# Verify the installation was successful
!erl -version

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,357 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,489 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,608 kB]
G

In [4]:
%%writefile sensor_hub.erl
% ==========================================
% 2. ERLANG ARCHITECTURE & ACTOR MODEL
% ==========================================
-module(sensor_hub).
-export([start/0, init_supervisor/0, supervisor/1, sensor/1]).

% --- Main Entry Point ---
start() ->
    io:format("🚀 Booting Erlang IoT Sensor Hub...~n"),

    % Spawn the master supervisor process
    SupPid = spawn(?MODULE, init_supervisor, []),

    % Tell the supervisor to spin up 3 concurrent sensors
    SupPid ! {start_sensor, "Temp_Zone_A"},
    SupPid ! {start_sensor, "Temp_Zone_B"},
    SupPid ! {start_sensor, "Temp_Zone_C"},

    % Let them run for a moment
    timer:sleep(2000),

    % INJECT CHAOS: Tell the supervisor to crash one of the sensors
    SupPid ! {crash_sensor, "Temp_Zone_B"},

    % Let the system recover and run a bit more
    timer:sleep(3000),
    io:format("🏁 Simulation complete. System survived.~n").

% --- Supervisor Logic ---
init_supervisor() ->
    % 'trap_exit' is the magic of Erlang. If a child process dies,
    % the supervisor gets a message instead of dying with it.
    process_flag(trap_exit, true),
    supervisor([]).

supervisor(Sensors) ->
    receive
        % Command to start a new sensor
        {start_sensor, Name} ->
            % spawn_link links the child's heartbeat to this supervisor
            Pid = spawn_link(?MODULE, sensor, [Name]),
            io:format("🛡️ Supervisor: Spun up sensor ~s [PID: ~p]~n", [Name, Pid]),
            supervisor([{Pid, Name} | Sensors]);

        % Command to simulate a fatal error
        {crash_sensor, Name} ->
            {Pid, _} = lists:keyfind(Name, 2, Sensors),
            io:format("~n⚡ INJECTING FAULT: Forcefully killing ~s...~n", [Name]),
            exit(Pid, kill),
            supervisor(Sensors);

        % The heartbeat monitor caught a death!
        {'EXIT', Pid, Reason} ->
            {Pid, Name} = lists:keyfind(Pid, 1, Sensors),
            io:format("🚨 ALERT: ~s crashed! Reason: ~p~n", [Name, Reason]),
            io:format("♻️ RECOVERY: Implementing 'Let it crash' philosophy. Restarting ~s...~n~n", [Name]),

            % Immediately resurrect the sensor
            NewPid = spawn_link(?MODULE, sensor, [Name]),

            % Update our state list with the new PID
            NewSensors = [{NewPid, Name} | lists:keydelete(Pid, 1, Sensors)],
            supervisor(NewSensors)
    end.

% --- Worker Logic (The Sensor) ---
sensor(Name) ->
    receive
        % Wait for incoming messages, otherwise do work in the 'after' block
        _ -> sensor(Name)
    after 1000 ->
        % Transmit data every 1 second
        io:format("📡 ~s: Transmitting telemetry data... OK~n", [Name]),
        sensor(Name)
    end.

Overwriting sensor_hub.erl


In [5]:
# ==========================================
# 3. COMPILE AND RUN THE SYSTEM
# ==========================================
# erlc compiles the code into a .beam file
# erl -noshell runs it, executes the start function, and stops the environment cleanly
!erlc sensor_hub.erl
!erl -noshell -s sensor_hub start -s init stop

\x{1F680} Booting Erlang IoT Sensor Hub...
\x{1F6E1}\x{FE0F} Supervisor: Spun up sensor Temp_Zone_A [PID: <0.78.0>]
\x{1F6E1}\x{FE0F} Supervisor: Spun up sensor Temp_Zone_B [PID: <0.79.0>]
\x{1F6E1}\x{FE0F} Supervisor: Spun up sensor Temp_Zone_C [PID: <0.80.0>]
\x{1F4E1} Temp_Zone_A: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zone_B: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zone_C: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zone_A: Transmitting telemetry data... OK

\x{26A1} INJECTING FAULT: Forcefully killing Temp_Zone_B...
\x{1F6A8} ALERT: Temp_Zone_B crashed! Reason: killed
\x{267B}\x{FE0F} RECOVERY: Implementing 'Let it crash' philosophy. Restarting Temp_Zone_B...

\x{1F4E1} Temp_Zone_C: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zone_A: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zone_B: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zone_C: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zone_A: Transmitting telemetry data... OK
\x{1F4E1} Temp_Zo